# 📉 Tema 20: Regresión Regularizada - Ridge y Lasso

¡Bienvenido/a a la optimización avanzada de modelos!

Cuando trabajamos con bases de datos que tienen decenas o cientos de columnas, la Regresión Lineal Múltiple estándar tiende a sufrir de **Sobreajuste (Overfitting)**: el modelo memoriza los datos en lugar de aprender el patrón real. Para evitar que nuestro algoritmo colapse ante datos nuevos, utilizamos la **Regularización**, una técnica matemática que "castiga" o "penaliza" a las variables que no aportan valor predictivo.

## 🚀 Contenido del Cuaderno

1. **Comprender la Regularización:** ¿Qué son las penalizaciones L1 y L2?
2. **Ejemplo Práctico de Alta Dimensionalidad:** Análisis de crímenes (UCI).
3. **Proyecto Práctico:** Predicción de Emisiones de $CO_2$ en vehículos.
4. **Regresión Ridge y Lasso:** Reduciendo la varianza y seleccionando variables.
5. **Nivel Pro:** Tuberías de Machine Learning (Pipelines) y OOP.
6. **Glosario Técnico y Conclusión.**

---
> **Instrucciones:** Ejecuta las celdas paso a paso. Asegúrate de tener el archivo `FuelConsumptionCo2.xlsx` en tu directorio.

## 1. Comprende Regresión Regularizada: Ridge y Lasso

En una regresión normal, el algoritmo ajusta los coeficientes ($\beta$) para que el error sea cero. En la regularizada, añadimos un "castigo" a la función matemática si esos coeficientes crecen demasiado. Este castigo es controlado por un hiperparámetro llamado **Alfa ($\alpha$)**.

* **Ridge (Regularización L2):** Penaliza el cuadrado de los coeficientes. No elimina variables, pero encoge sus coeficientes casi a cero, haciéndolas menos influyentes. Es excelente para lidiar con columnas altamente correlacionadas.
* **Lasso (Regularización L1):** Penaliza el valor absoluto de los coeficientes. Su "superpoder" es que puede reducir los coeficientes de variables inútiles **exactamente a cero**. En la práctica, esto significa que Lasso elimina columnas automáticamente.

## 2. Estudia el ejemplo práctico (Alta Dimensionalidad)

Para entender por qué necesitamos la Regularización, observemos la base de datos *Communities and Crime* de la Universidad de California (UCI). Esta base tiene más de 120 variables para intentar predecir el número de crímenes violentos por población (`ViolentCrimesPerPop`). 

Si aplicamos una regresión estándar, el modelo intentará darle peso matemático a columnas que no aportan nada, generando mucho "ruido" y Overfitting. 

**¿La solución? Regresión Lasso (L1).**
Lasso tiene un "superpoder": si detecta que una variable no es útil para predecir el crimen, **reduce su coeficiente exactamente a cero**, eliminándola automáticamente del modelo. Veamos esto en código.

In [16]:
# --- Ejemplo Práctico: Base de datos de Crímenes (UCI) ---
import pandas as pd 
import numpy as np 
from sklearn.linear_model  import LassoCV
from sklearn.preprocessing import StandardScaler

# 1. Cargamos los datos directamente desde el repositorio de UCI
url_crimen = 'http://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data'
# La base no tiene cabeceras y usa '?' para los valores nulos
df_crimen = pd.read_csv(url_crimen, header=None, na_values='?')

# 2. Limpieza rápida: Eliminamos columnas con nulos y variables de texto/identificadores
df_crimen_limpio = df_crimen.dropna(axis=1).select_dtypes(include=[np.number])

# La variable a predecir (ViolentCrimesPerPop) es la última columna del dataset
X_crimen = df_crimen_limpio.iloc[:, :-1]
y_crimen = df_crimen_limpio.iloc[:, -1]

# 3. Estandarizamos los datos (Obligatorio para Regresión Regularizada)
scaler_crimen = StandardScaler()
X_crimen_scaled = scaler_crimen.fit_transform(X_crimen)

# 4. Aplicamos LassoCV para que busque el Alpha óptimo y elimine columnas inútiles
modelo_lasso_crimen = LassoCV(cv=5, random_state=42)
modelo_lasso_crimen.fit(X_crimen_scaled, y_crimen)

# Contamos cuántas variables "sobrevivieron" al castigo de Lasso
coeficientes = modelo_lasso_crimen.coef_
variables_totales = len(coeficientes)
variables_eliminadas = np.sum(coeficientes == 0)
variables_utiles = np.sum(coeficientes != 0)

print("=== Análisis de Alta Dimensionalidad con Lasso ===")
print(f"Total de variables originales: {variables_totales}")
print(f"Variables ELIMINADAS por Lasso (Coeficiente = 0): {variables_eliminadas}")
print(f"Variables útiles que sobrevivieron: {variables_utiles}")
print("\nConclusión: Lasso nos salvó de un modelo ruidoso eliminando matemáticamente las columnas que no aportaban valor predictivo.")

=== Análisis de Alta Dimensionalidad con Lasso ===
Total de variables originales: 101
Variables ELIMINADAS por Lasso (Coeficiente = 0): 27
Variables útiles que sobrevivieron: 74

Conclusión: Lasso nos salvó de un modelo ruidoso eliminando matemáticamente las columnas que no aportaban valor predictivo.


---
## 📝 Proyecto Práctico: Emisiones de $CO_2$

Ahora aplicaremos estos conceptos a nuestro problema principal. Construiremos modelos predictivos usando la base de datos de consumo de combustible. 

**Objetivos:**
1. Cargar el archivo Excel, eliminar datos categóricos y limpiar valores nulos.
2. Aplicar Regresión Lineal Múltiple y comparar el pronóstico manual vs. la función `.predict()`.
3. Aplicar Regresión Ridge buscando el Alpha óptimo.
4. Aplicar Regresión Lasso buscando el Alpha óptimo.
5. Seleccionar el mejor modelo evaluando $R^2$ y Error Absoluto Medio (MAE).

In [28]:
# --- Resolución del Proyecto Práctico ---
from sklearn. model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_absolute_error
import warnings 
warnings.filterwarnings('ignore')

# 1. Cargar y Limpiar la Base de Datos (USANDO READ_EXCEL)
df= pd.read_excel('FuelConsumptionCo2.xlsx')

# Verificar valores nulos
nulos_totales = df.isnull().sum().sum()
print(f"Valores nulos en el dataset: {nulos_totales}")
if nulos_totales > 0:
    df= df.dropna()
    print("Los valores nulos fueron eliminados.")

# Identificamos la variable objetivo
y = df ['CO2EMISSIONS']

# Eliminamos variables categóricas (texto) y la variable objetivo
columnas_categoricas = ['MAKE', 'MODEL', 'VEHICLECLASS', 'TRANSMISSION', 'FUELTYPE']
X = df.drop(columns=columnas_categoricas + ['CO2EMISSIONS'])

# Dividimos en datos de entrenamiento y prueba (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# MUY IMPORTANTE: La regularizacion exige que los datso estén Estandarizados (Escalados)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Datos limpios, separados y estandarizados con éxito.")
print(f"Variables independientes remanentes para entrenar: {list(X.columns)}")

Valores nulos en el dataset: 0

✅ Datos limpios, separados y estandarizados con éxito.
Variables independientes remanentes para entrenar: ['MODELYEAR', 'ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_CITY', 'FUELCONSUMPTION_HWY', 'FUELCONSUMPTION_COMB', 'FUELCONSUMPTION_COMB_MPG']


### Paso 2: Regresión Múltiple Tradicional y Predicción Manual
Ajustaremos el modelo estándar y entenderemos qué ocurre matemáticamente bajo el capó de la función `.predict()`.

In [18]:
# --- Regresión Lineal Múltiple ---
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

# Métricas
y_pred_lin = lin_reg.predict(X_test_scaled)
r2_lin = r2_score(y_test, y_pred_lin)
mae_lin = mean_absolute_error(y_test, y_pred_lin)

print("=== 1. REGRESIÓN LINEAL MÚLTIPLE ===")
print(f"R^2: {r2_lin:.4f}")
print(f"MAE: {mae_lin:.4f}")

# --- Comprobación Manual vs Función predict() ---
# Tomamos la primera observación de prueba
primera_observacion = X_test_scaled[0]

# Predicción usando la instrucción del modelo
pred_instruccion = lin_reg.predict([primera_observacion])[0]

# Predicción manual: Intercepto + (Coeficiente_1 * X_1) + ... + (Coeficiente_n * X_n)
pred_manual = lin_reg.intercept_ + np.dot(lin_reg.coef_, primera_observacion)

print("\n--- Comprobación de Predicción (Observación 0) ---")
print(f"Resultado con .predict(): {pred_instruccion:.4f}")
print(f"Resultado matemático manual: {pred_manual:.4f}")
print("¿Coinciden? SÍ. La instrucción .predict() simplemente automatiza la suma de los productos cruzados entre los datos y los coeficientes descubiertos por el modelo.")

=== 1. REGRESIÓN LINEAL MÚLTIPLE ===
R^2: 0.9005
MAE: 7.3391

--- Comprobación de Predicción (Observación 0) ---
Resultado con .predict(): 242.9269
Resultado matemático manual: 242.9269
¿Coinciden? SÍ. La instrucción .predict() simplemente automatiza la suma de los productos cruzados entre los datos y los coeficientes descubiertos por el modelo.


### Paso 3 y 4: Regresión Ridge y Regresión Lasso (con Alpha Óptimo)
En lugar de adivinar el parámetro Alpha ($\alpha$), utilizaremos las funciones `RidgeCV` y `LassoCV`, las cuales aplican **Validación Cruzada** para probar docenas de valores de Alpha automáticamente y quedarse con el que minimice el error.

In [20]:
# --- 2. Regresión Ridge (L2) con búsqueda de Alpha ---
alphas_prueba = np.logspace(-3, 2, 100)
ridge_cv = RidgeCV(alphas=alphas_prueba, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_cv.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print("=== 2. REGRESIÓN RIDGE (L2) ===")
print(f"Alpha Óptimo encontrado: {ridge_cv.alpha_:.4f}")
print(f"R^2: {r2_ridge:.4f}")
print(f"MAE: {mae_ridge:.4f}\n")

# --- 3. Regresión Lasso (L1) con búsqueda de Alpha ---
lasso_cv = LassoCV(alphas=alphas_prueba, cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

y_pred_lasso = lasso_cv.predict(X_test_scaled)
r2_lasso = r2_score(y_test, y_pred_lasso)
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)

print("=== 3. REGRESIÓN LASSO (L1) ===")
print(f"Alpha Óptimo encontrado: {lasso_cv.alpha_:.4f}")
print(f"R^2: {r2_lasso:.4f}")
print(f"MAE: {mae_lasso:.4f}")

=== 2. REGRESIÓN RIDGE (L2) ===
Alpha Óptimo encontrado: 13.8489
R^2: 0.9030
MAE: 7.4825

=== 3. REGRESIÓN LASSO (L1) ===
Alpha Óptimo encontrado: 0.0231
R^2: 0.9005
MAE: 7.3414


### 💡 Análisis: ¿Cuál de los 3 modelos resultó ser el mejor?

Al analizar los resultados, **los tres modelos arrojaron métricas casi idénticas**. 

**Explicación detallada:**
Cuando la Regresión Lineal Múltiple, Ridge y Lasso arrojan el mismo resultado, significa que **nuestra base de datos no sufre de Overfitting ni de multicolinealidad severa**. Las variables numéricas que dejamos en el modelo (Tamaño del motor, Cilindros, Consumo) son excelentes predictores de las Emisiones de $CO_2$ por naturaleza y no incluyen ruido.

Sin embargo, en un entorno de producción estricto, **Lasso** o **Ridge** siempre serán preferibles sobre la Regresión Lineal Estándar. Nos protegen a futuro: si el día de mañana alguien inyecta 50 variables "basura" a nuestra base de datos, el modelo Lineal colapsaría, mientras que Lasso reduciría los coeficientes de esa basura a $0$ de manera automática.

## 🚀 Nivel Pro: Evaluador de Modelos Orientado a Objetos

El código anterior cumple con la práctica, pero escribir el entrenamiento, la predicción y el cálculo de métricas tres veces distintas no es eficiente (*Violación del principio DRY: Don't Repeat Yourself*). 

A continuación, crearemos una clase en Python que ingiere nuestros datos, los escala automáticamente y evalúa los tres algoritmos, devolviéndonos una tabla comparativa limpia.

In [21]:
class EvaluadorRegresion:
    """
    Clase profesional para evaluar y comparar modelos de regresión estándar y regularizada.
    Aplica buenas prácticas de Machine Learning como el escalado automático seguro.
    """
    def __init__(self):
        self.scaler = StandardScaler()
        self.modelos = {
            "Lineal Múltiple": LinearRegression(),
            "Ridge (L2)": RidgeCV(alphas=np.logspace(-2, 2, 50), cv=5),
            "Lasso (L1)": LassoCV(alphas=np.logspace(-2, 2, 50), cv=5, max_iter=10000)
        }
        self.resultados = []

    def fit_evaluate(self, X_tr: pd.DataFrame, X_te: pd.DataFrame, y_tr: pd.Series, y_te: pd.Series):
        # 1. Pipeline Interno: Escalar datos (Evitando Data Leakage)
        X_tr_sc = self.scaler.fit_transform(X_tr)
        X_te_sc = self.scaler.transform(X_te)

        # 2. Iterar sobre el diccionario de modelos
        for nombre, modelo in self.modelos.items():
            modelo.fit(X_tr_sc, y_tr)
            predicciones = modelo.predict(X_te_sc)
            
            # Recopilación de métricas
            metricas = {
                "Modelo": nombre,
                "R^2": round(r2_score(y_te, predicciones), 4),
                "MAE": round(mean_absolute_error(y_te, predicciones), 4),
                "Alpha Óptimo": round(modelo.alpha_, 4) if hasattr(modelo, 'alpha_') else "N/A"
            }
            self.resultados.append(metricas)
            
        return pd.DataFrame(self.resultados).set_index("Modelo")

# --- Automatización de Pruebas ---
if __name__ == "__main__":
    print("=== 🚀 Pipeline de Evaluación de Modelos ===")
    evaluador = EvaluadorRegresion()
    
    # Inyectamos los datos separados en el paso 1 (Crudos, la clase los escala)
    tabla_comparativa = evaluador.fit_evaluate(X_train, X_test, y_train, y_test)
    display(tabla_comparativa)

=== 🚀 Pipeline de Evaluación de Modelos ===


,R^2,MAE,Alpha Óptimo
Modelo,,,
Lineal Múltiple,0.9005,7.3391,N/A
Ridge (L2),0.9032,7.4958,15.2642
Lasso (L1),0.9005,7.3414,0.045


## 🔍 Explicación de la Versión Profesional

1. **Diccionario de Modelos:** En lugar de repetir código, almacenamos las instancias de Scikit-Learn en un diccionario (`self.modelos`). Esto nos permite usar un bucle `for` para entrenarlos todos con las mismas exactas condiciones.
2. **Transformación Segura (`scaler.fit_transform` vs `scaler.transform`):** Nota cómo el escalador solo "aprende" (hace *fit*) de los datos de entrenamiento (`X_tr`), y luego solo "aplica" ese aprendizaje a los datos de prueba (`X_te`). Hacer `fit` sobre los datos de prueba es un error gravísimo en ML llamado *Data Leakage* (Fuga de datos), el cual hemos evitado exitosamente aquí.
3. **Manejo Dinámico de Atributos (`hasattr`):** Usamos la instrucción de Python `hasattr()` para preguntar: *"¿Este algoritmo tiene un parámetro Alpha?"*. Si lo tiene (Ridge/Lasso), lo imprime; si no lo tiene (Lineal Estándar), imprime "N/A", evitando que el código colapse.

## 📚 Glosario de Regresión Regularizada

Para hablar y codificar como un Científico de Datos Senior, es indispensable dominar estos términos:

* **Alfa ($\alpha$):** Parámetro de regularización que controla la "fuerza" del castigo matemático. Si $\alpha = 0$, equivale a una regresión estándar. Valores muy altos pueden subajustar el modelo.
* **Escalado / Estandarización:** Proceso crítico antes de aplicar Ridge o Lasso. Ajusta todas las características para que tengan una media de cero y desviación estándar de uno. Si no se hace, las penalizaciones matemáticas serían injustas para las variables con números grandes.
* **LassoCV / RidgeCV:** Variantes en *Scikit-learn* que utilizan **Validación cruzada** interna para buscar y seleccionar automáticamente el mejor valor de Alfa.
* **Normalización:** Técnica de escalado que ajusta los valores estrictamente a un rango específico (usualmente de 0 a 1).
* **Penalización:** El término añadido a la función de costo que impone una restricción sobre el tamaño de los coeficientes, previniendo el sobreajuste.
* **Regularización L1 (Lasso):** Utiliza la suma de valores absolutos para penalizar. Elimina automáticamente las variables irrelevantes volviendo su coeficiente cero.
* **Regularización L2 (Ridge):** Minimiza la suma de los cuadrados de los coeficientes. Distribuye la penalización para lidiar con multicolinealidad, pero rara vez elimina variables por completo.
* **Scikit-learn:** Biblioteca estándar de Python utilizada para implementar algoritmos de Machine Learning en producción.
* **Validación cruzada (Cross-Validation):** Método robusto para evaluar modelos diviendo los datos en subconjuntos y rotándolos (ej. entrenar en 4, probar en 1), ayudando a seleccionar el mejor hiperparámetro sin sesgos.

# 🎉 Conclusión del Cuaderno: Modelos a Prueba de Balas

¡Excelente trabajo finalizando este tema avanzado!

El Machine Learning no se trata de quién usa el algoritmo más complejo, sino de quién controla mejor los errores. A través de este módulo hemos aprendido que:

### 🧠 Resumen Técnico:
* Una Regresión Lineal estándar buscará encajar perfectamente en los datos de entrenamiento, pero a menudo falla terriblemente ante datos nuevos (Overfitting).
* La **Regularización (Ridge y Lasso)** es nuestra póliza de seguro. Le enseñamos a la computadora a ser "escéptica" respecto a variables que parecen importar demasiado.
* Comprobamos, a través del análisis de *Crímenes de la UCI*, cómo **Lasso (L1)** nos sirve como una herramienta automática de limpieza de datos, apagando matemáticamente las columnas irrelevantes.
* Entendimos la importancia crítica de la **Estandarización** para que los algoritmos de castigo funcionen de manera justa y equilibrada.
